# SCS Return Period Exploration

Distribution of NWS flood-threshold return periods across US Stream Channel Survey (SCS) basin attributes.

Variables explored:
- **Categorical:** StreamOrde, SizeClass, GradientClass, Confinement, Maheu_class, Bif_Class, Div_Class
- **Continuous:** RL, VBA, RWA, CatArea, VBL, VBL_RL_R, VBA_RWA_R, countUp, countDown, orderup1, JulAug_tempC

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.nonparametric.smoothers_lowess as lowess_sm

import matplotlib.font_manager as fm
font_path = '/home/ryan/data/assets/fonts/FiraSans/FiraSans-Regular.ttf'
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'Fira Sans'


DATA_DIR = Path("/home/ryan/data/flood_hazard/ffa")
META_DIR = Path("/home/ryan/data/flood_hazard/metadata")
NHD_DIR  = Path("/home/ryan/data/flood_hazard/NHD/US SCS")

plt.rcParams['figure.dpi'] = 150
sns.set_theme(style='whitegrid', palette='muted')

RP_COLS    = ['action_return_period_yr', 'flood_return_period_yr',
              'moderate_return_period_yr', 'major_return_period_yr']
THRESHOLDS = ['action', 'flood', 'moderate', 'major']
THR_LABELS = ['Action', 'Flood', 'Moderate', 'Major']

## 1. Load core data & QC filter

In [ ]:
ffa  = pd.read_parquet(DATA_DIR / "flood_frequency.parquet").drop_duplicates('site_no')
ppcc = pd.read_parquet(DATA_DIR / "ppcc.parquet")

ffa[RP_COLS] = ffa[RP_COLS].replace(np.inf, np.nan)

ok_sites = ppcc.loc[ppcc['ppcc_ok'], 'site_no']
ffa_ok = ffa[
    (ffa['record_ok'] == True) &
    (ffa['degenerate_fit'] == False) &
    ffa['site_no'].isin(ok_sites)
].copy()

gm = pd.read_parquet(META_DIR / "gage_map.parquet", columns=['site_no', 'reach_id'])
df = ffa_ok[['site_no'] + RP_COLS].merge(gm, on='site_no', how='left')
df['COMID'] = pd.to_numeric(df['reach_id'], errors='coerce').astype('Int64')

for t in THRESHOLDS:
    df[f'log_{t}_rp'] = np.log10(df[f'{t}_return_period_yr'])

print(f"QC-pass sites : {len(df):,}")
print(f"With COMID    : {df['COMID'].notna().sum():,}")

## 2. Load SCS tables

In [ ]:
def load_nhd_table(files_spec, keep_cols):
    """Load and concat NHD regional CSV files after renaming columns.
    files_spec: list of (path, rename_dict)
    """
    parts = []
    for path, renames in files_spec:
        d = pd.read_csv(path).rename(columns=renames)
        parts.append(d[[c for c in keep_cols if c in d.columns]])
    combined = pd.concat(parts, ignore_index=True)
    combined['COMID'] = pd.to_numeric(combined['COMID'], errors='coerce').astype('Int64')
    return combined.drop_duplicates('COMID')

In [ ]:
SG = NHD_DIR / "Size_Gradient"
VC = NHD_DIR / "Valley_Confinement"
TM = NHD_DIR / "Temperature"
BN = NHD_DIR / "Bifurcation_Network"

# ── Size_Gradient ─────────────────────────────────────────────────────────────
sg_rename = {
    'StreamOrder': 'StreamOrde',
    'Size_Class':  'SizeClass',
    'Gradient_Class': 'GradientClass',
}
nhd_sg = load_nhd_table([
    (SG / "East_SizeGradient.csv", sg_rename),
    (SG / "LM_SizeGradient.csv",   {}),
    (SG / "UM_SizeGradient.csv",   {'StreamOrder': 'StreamOrde'}),
    (SG / "West_SizeGradient.csv", {}),
], keep_cols=['COMID', 'StreamOrde', 'SizeClass', 'GradientClass'])

# ── Valley_Confinement ────────────────────────────────────────────────────────
# East uses RWA; LM/UM/West use SWA → normalise to RWA
nhd_vc = load_nhd_table([
    (VC / "East_VC.csv", {}),
    (VC / "LM_VC.csv",   {'SWA': 'RWA'}),
    (VC / "UM_VC.csv",   {'SWA': 'RWA'}),
    (VC / "West_VC.csv", {'SWA': 'RWA'}),
], keep_cols=['COMID', 'RL', 'VBA', 'RWA', 'CatArea', 'VBL', 'VBL_RL_R', 'VBA_RWA_R', 'Confinement'])
nhd_vc = nhd_vc[nhd_vc['Confinement'].isin(['Unconfined', 'Mod Confined', 'Confined'])].copy()
for col in ['RL', 'VBA', 'RWA', 'CatArea', 'VBL', 'VBL_RL_R', 'VBA_RWA_R']:
    nhd_vc[col] = pd.to_numeric(nhd_vc[col], errors='coerce')

# ── Temperature ───────────────────────────────────────────────────────────────
nhd_tm = load_nhd_table([
    (TM / "East_temp.csv", {'Maheu_type': 'Maheu_class'}),
    (TM / "LM_temp.csv",   {}),
    (TM / "UM_temp.csv",   {'JulyAug_tempC': 'JulAug_tempC'}),
    (TM / "West_temp.csv", {}),
], keep_cols=['COMID', 'Maheu_class', 'JulAug_tempC'])
nhd_tm['JulAug_tempC'] = pd.to_numeric(nhd_tm['JulAug_tempC'], errors='coerce')

# ── Bifurcation_Network ───────────────────────────────────────────────────────
# LM uses orderUp1 and Bif_order instead of orderup1 and Bif_Class
nhd_bn = load_nhd_table([
    (BN / "East_BifNetwork.csv",  {}),
    (BN / "LM_BifNetwork.csv",   {'orderUp1': 'orderup1', 'Bif_order': 'Bif_Class'}),
    (BN / "UM_BifNetwork.csv",   {}),
    (BN / "West_BifNetwork.csv", {}),
], keep_cols=['COMID', 'countUp', 'countDown', 'orderup1', 'Bif_Class', 'Div_Class'])
for col in ['countUp', 'countDown', 'orderup1']:
    nhd_bn[col] = pd.to_numeric(nhd_bn[col], errors='coerce')

# ── Join onto df ──────────────────────────────────────────────────────────────
n_comid = df['COMID'].notna().sum()
for name, tbl, probe in [
    ('SizeGradient',       nhd_sg, 'StreamOrde'),
    ('ValleyConfinement',  nhd_vc, 'Confinement'),
    ('Temperature',        nhd_tm, 'Maheu_class'),
    ('BifurcationNetwork', nhd_bn, 'Bif_Class'),
]:
    df = df.merge(tbl, on='COMID', how='left')
    print(f"{name:22s}  {df[probe].notna().sum():,} / {n_comid:,}")

## 3. Coverage report

In [ ]:
SCS_VARS = [
    'StreamOrde', 'SizeClass', 'GradientClass',
    'Confinement',
    'Maheu_class', 'JulAug_tempC',
    'Bif_Class', 'Div_Class',
    'RL', 'VBA', 'RWA', 'CatArea', 'VBL', 'VBL_RL_R', 'VBA_RWA_R',
    'countUp', 'countDown', 'orderup1',
]

cov = pd.DataFrame({
    'n_valid': [df[c].notna().sum() for c in SCS_VARS],
    'pct':     [round(df[c].notna().mean() * 100, 1) for c in SCS_VARS],
}, index=SCS_VARS).sort_values('pct', ascending=False)
print(cov.to_string())

df_scs = df[df['COMID'].notna() &
            df[['StreamOrde', 'Confinement', 'Maheu_class']].notna().any(axis=1)].copy()
print(f"\nSites with at least one SCS attribute: {len(df_scs):,}")

## 4. Categorical variables — boxplots

In [ ]:
def cat_boxplot_grid(df, cat_col, title=None, cat_order=None, figsize=(16, 4)):
    sub = df[[cat_col] + RP_COLS].dropna(subset=[cat_col])
    if cat_order is None:
        medians = sub.groupby(cat_col)[RP_COLS[0]].median()
        cat_order = medians.sort_values().index.tolist()
    # drop any categories with no data
    cat_order = [c for c in cat_order if c in sub[cat_col].values]

    fig, axes = plt.subplots(1, 4, figsize=figsize, sharey=True)
    for ax, rp_col, lbl in zip(axes, RP_COLS, THR_LABELS):
        data = [sub.loc[sub[cat_col] == cat, rp_col].dropna().values for cat in cat_order]
        ax.boxplot(
            data,
            patch_artist=True,
            flierprops=dict(marker='.', markersize=2, alpha=0.3),
            medianprops=dict(color='black', lw=1.5),
        )
        ax.set_yscale('log')
        ax.set_xticks(range(1, len(cat_order) + 1))
        ax.set_xticklabels(
            [f"{c}\n(n={len(d)}" + ")" for c, d in zip(cat_order, data)],
            fontsize=7, rotation=30, ha='right',
        )
        ax.set_title(lbl)
        if ax is axes[0]:
            ax.set_ylabel('Return period (yr)')
            
        ax.set_ylim(0, 1000)

    fig.suptitle(title or cat_col, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# # Stream order (integer sort)
# stream_order_vals = sorted(df_scs['StreamOrde'].dropna().astype(int).unique())
# cat_boxplot_grid(df_scs, 'StreamOrde',
#                  cat_order=[str(v) for v in stream_order_vals],
#                  title='Return Period by Stream Order')

In [ ]:
cat_boxplot_grid(df_scs, 'SizeClass', title='Return Period by Size Class')

In [ ]:
grad_order = ['Very low', 'Low', 'Moderate', 'Moderate High', 'High', 'Steep']
cat_boxplot_grid(df_scs, 'GradientClass',
                 cat_order=grad_order,
                 title='Return Period by Gradient Class')

In [ ]:
cat_boxplot_grid(df_scs, 'Confinement',
                 cat_order=['Unconfined', 'Mod Confined', 'Confined'],
                 title='Return Period by Valley Confinement')

In [ ]:
cat_boxplot_grid(df_scs, 'Maheu_class', title='Return Period by Maheu Thermal Class')

In [ ]:
# Limit Bif_Class to top-8 by count — many small classes otherwise
top_bif = df_scs['Bif_Class'].value_counts().head(8).index.tolist()
cat_boxplot_grid(df_scs[df_scs['Bif_Class'].isin(top_bif)], 'Bif_Class',
                 title='Return Period by Bifurcation Class (top 8)')

In [ ]:
cat_boxplot_grid(df_scs, 'Div_Class', title='Return Period by Divergence Class')

## 5. Continuous variables — scatter + LOWESS

In [ ]:
def cont_scatter_grid(df, pred_col, xlabel, log_x=True, frac=0.3, figsize=(16, 4)):
    fig, axes = plt.subplots(1, 4, figsize=figsize)
    for ax, rp_col, lbl in zip(axes, RP_COLS, THR_LABELS):
        sub = df[[pred_col, rp_col]].dropna()
        sub = sub[sub[rp_col] > 0]
        if log_x:
            sub = sub[sub[pred_col] > 0]
            xvals = np.log10(sub[pred_col])
            ax.set_xlabel(f'log₁₀({xlabel})')
        else:
            xvals = sub[pred_col]
            ax.set_xlabel(xlabel)
        yvals = np.log10(sub[rp_col])

        ax.scatter(xvals, yvals, s=5, alpha=0.35, color='steelblue', rasterized=True)

        if len(sub) > 20:
            order = xvals.argsort()
            smoothed = lowess_sm.lowess(
                yvals.values[order], xvals.values[order],
                frac=frac, return_sorted=True,
            )
            ax.plot(smoothed[:, 0], smoothed[:, 1], color='black', lw=1.5)

        if len(sub) > 10:
            rho, _ = stats.spearmanr(xvals, yvals)
            ax.text(0.97, 0.03, f'r={rho:.2f}', transform=ax.transAxes,
                    ha='right', va='bottom', fontsize=8)

        ax.set_title(lbl)
        if ax is axes[0]:
            ax.set_ylabel('log₁₀(Return period, yr)')

    fig.suptitle(f'Return Period vs. {pred_col}', fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
cont_scatter_grid(df_scs, 'RL',        'RL (reach length)',        log_x=True)
cont_scatter_grid(df_scs, 'VBA',       'VBA (valley bottom area)', log_x=True)
cont_scatter_grid(df_scs, 'RWA',       'RWA (riparian width area)', log_x=True)
cont_scatter_grid(df_scs, 'CatArea',   'CatArea (catchment area)', log_x=True)
cont_scatter_grid(df_scs, 'VBL',       'VBL (valley bottom length)', log_x=True)
cont_scatter_grid(df_scs, 'VBL_RL_R',  'VBL/RL ratio',             log_x=True)
cont_scatter_grid(df_scs, 'VBA_RWA_R', 'VBA/RWA ratio',            log_x=True)

In [ ]:
cont_scatter_grid(df_scs, 'JulAug_tempC', 'Jul-Aug temp (°C)', log_x=False)
cont_scatter_grid(df_scs, 'countUp',    'countUp (upstream junctions)',    log_x=False)
cont_scatter_grid(df_scs, 'countDown',  'countDown (downstream junctions)', log_x=False)
cont_scatter_grid(df_scs, 'orderup1',   'orderup1 (upstream order)',        log_x=False)

## 6. Spearman correlation summary

In [ ]:
LOG_CONT = ['RL', 'VBA', 'RWA', 'CatArea', 'VBL', 'VBL_RL_R', 'VBA_RWA_R']
LIN_CONT = ['JulAug_tempC', 'countUp', 'countDown', 'orderup1']

for col in LOG_CONT:
    df_scs[f'log_{col}'] = np.log10(df_scs[col].clip(lower=1e-9))

pred_cols = [f'log_{c}' for c in LOG_CONT] + LIN_CONT
log_rp_cols = [f'log_{t}_rp' for t in THRESHOLDS]

rows = []
for pred in pred_cols:
    row = {'predictor': pred}
    abs_rs = []
    for tc, t in zip(log_rp_cols, THRESHOLDS):
        pair = df_scs[[pred, tc]].dropna()
        r = stats.spearmanr(pair[pred], pair[tc])[0] if len(pair) > 10 else np.nan
        row[t] = round(r, 3) if not np.isnan(r) else np.nan
        abs_rs.append(abs(r) if not np.isnan(r) else 0)
    row['mean_|r|'] = round(np.nanmean(abs_rs), 3)
    rows.append(row)

summary = (pd.DataFrame(rows)
             .set_index('predictor')
             .sort_values('mean_|r|', ascending=False))
print(summary.to_string())

fig, ax = plt.subplots(figsize=(6, 6))
sns.heatmap(
    summary[THRESHOLDS],
    annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.3, ax=ax, cbar_kws={'shrink': 0.7},
)
ax.set_title('Spearman r: SCS Continuous Attributes vs. Return Periods', fontweight='bold')
plt.tight_layout()
plt.show()